In [1]:
import matplotlib
%matplotlib tk

import matplotlib.pyplot as plt
%autosave 180
%load_ext autoreload
%autoreload 2
import numpy as np

from scipy import ndimage as ndi
from skimage.segmentation import watershed
from skimage.feature import peak_local_max
import scipy
import os
import time

#
from utils.utils import ComputeROIs

Autosaving every 180 seconds


In [4]:
#######################################################################
########### LOAD PRE-BMI DATA (e.g. 10-15mins recording) ##############
#######################################################################
fname = r'C:\code\test_data\image_10000frames.raw'
#fname = '/home/cat/data/donato/bscope_tests/image_10000frames.raw'
#fname = '/media/cat/4TBSSD/donato/Bscope_tests/image_10000frames.raw'
#fname = '/media/cat/4TB/donato/DON-008498/mouse1_calibration/Image_001_001.raw'

# 
gr = ComputeROIs(fname)



### GAUSIAN SMOOTHIN STEP TO REMOVE PHOTON SHOT NOISE IS THE SLOWEST 
### TODO: try to speed up to go faster plus use more data
###   - NOTE: The more data we use here, the better the cell detection etc.

In [9]:
#######################################################################
########### COMPUTE STD OVER TIME TO GET CELL FOOTPRINTS ##############
#######################################################################
# TODO: the gaussian filter step takes a long time
#      - try to speed up
#      - also, corelation map might be even better method for detecting ROIs
#         e.g. see suite2p's visualization tool <- seems pretty quick

# Also, need to exponse some more variables here like imshow vmin/vmax which are used to plot the final result
start = time.time()
#
gr.subsample = 5     # for std computation downsample to every N'th frame; the more frames the better the rois;
                      #   TODO: use correlation instead?! might be much faster; it is quit fast in other implemenations

#
gr.vmin = 200
gr.vmax = 400

#    
std_map = gr.make_std_map()
print ("total processing time: ", time.time()-start, " sec")

memmap :  (10000, 512, 512)
data into analysis:  (2000, 512, 512)
 gaussian filter width:  1 , order:  0
done filtering... (TO CHECK which axis are we filtering!!)
staring computing std...
done computing std...
total processing time:  31.82008123397827  sec


In [10]:
#######################################################################
########### RUN WATERSHED ALGORITHM DETECTION ON STD MAP ##############
#######################################################################
gr.find_roi_boundaries(std_map)

#
gr.scale=4000                      # spacing between each neuron trace because they are not normalized to the max vlaue
gr.trace_subsample = 1             # Subsample the time series to go faster;

# visualize traces
gr.compute_traces2()

memmap :  (10000, 512, 512)


100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████| 10000/10000 [00:11<00:00, 845.31it/s]


In [11]:
###############################################################
########### SELECT CELLS TO BE USED FOR ENSEMBLES #############
###############################################################

# save ensemble rois
# DON 8499 MOUSE
#gr.ensemble1 = [13,27]
#gr.ensemble2 = [43,3]

#
gr.ensemble1 = [2,3]
gr.ensemble2 = [5,10 ]

both = np.hstack((gr.ensemble1, gr.ensemble2))

# 
gr.scale=4000                      # spacing between each neuron trace because they are not normalized to the max vlaue

#
gr.show_traces_ids(both)


In [10]:
#########################################################################################
########### OPTIONAL - DEBUG OUTPUT FROM BMI_RUN TO COMPARE WITH CALBRATION #############
#########################################################################################
fname = '/home/cat/code/bmi/mouse1_calibration/bmi_results.npz'
data = np.load(fname, allow_pickle=True)

rois_raw = data['rois_traces_raw']
print (rois_raw.shape)

fig=plt.figure()
for k in range(rois_raw.shape[0]):
    temp = rois_raw[k]
    temp = temp- np.median(temp)
    
    plt.plot(temp+k*100)
    
plt.show()



(4, 20430)


In [29]:
#######################################################################
#######################################################################
#######################################################################
print (both)
print (len(gr.indexes))
fig = plt.figure()
for cell_id in both:
    gr.show_contour_map(std_map, 
                        [gr.indexes[cell_id]],
                        fig)
    
plt.show()




[13 27 43  3]
50


In [64]:
import mmap

    
fname_fluorescence = '/home/cat/code/bmi/mouse1_calibration/Image_001_001.raw'

fp = open(fname_fluorescence, "r")
n_frames_to_be_acquired=20000
image_width = 512
image_length = 512
byte_length = 2
byts = n_frames_to_be_acquired*image_width*image_length*byte_length
newfp = mmap.mmap(fp.fileno(), 
                  0, 
                  access=mmap.ACCESS_READ)

#
mview = memoryview(newfp)
print (" memroy view: ", mview)
newfp = np.asarray(mview, dtype=np.uint16)
print ("newfp: ", newfp.shape, type(newfp[0]))

# 
newfp = newfp.reshape(n_frames_to_be_acquired,512,512)
print (" final array view: ", newfp.shape)
print (type(newfp[0][0][0]))


# newfp = np.memmap(fname_fluorescence, 
#                        dtype='uint16', 
#                        mode='r',
#                        shape=n_frames_to_be_acquired*512*512).reshape(n_frames_to_be_acquired,512,512)

print (newfp.shape)
       
fig=plt.figure()
plt.imshow(newfp[100])
plt.show()

 memroy view:  <memory at 0x7f4c9df38400>
newfp:  (10485760000,) <class 'numpy.uint16'>


ValueError: cannot reshape array of size 10485760000 into shape (20000,512,512)

In [46]:
# loop over the remaning cells on the last frame 'z'

cell_id = both[0]
rois_traces_raw = []
rois_pixels = np.array(gr.indexes[cell_id])

# new way use exact pixel location
from tqdm import trange
for t in trange(n_frames_to_be_acquired):
    temp = newfp[t]              # take most recent frame
    temp = temp[rois_pixels.T[:, 0],        # broadcast/index into the frame as per ROI pixels
                rois_pixels.T[:, 1]]

    # divide by the number of pixels in the ROI - NOT SURE IF THIS IS CORRECT?!
    # TODO: these algorithms must match the default water disposal algorithms
    # TODO: USE A FUNCTION OVER THIS AND FOLLOWING STEP THAT IS SHARED WITH CALIBRATION CODE
    roi_sum0 = temp / rois_pixels[0].shape[0]

    # sum
    # TODO: not sure this is the correct function; to check literature
    # TODO: also this part shoudl be refactored to a callabale function by both calibration and BMI classes
    roi_sum0 = np.nansum(roi_sum0)

    # Note: Do not remove baseline yet; this is done in the smoothing step;
    # TODO: make sure that this approach is correct
    rois_traces_raw.append(roi_sum0)

fig=plt.figure()
plt.plot(rois_traces_raw)
plt.show()

100%|██████████| 20000/20000 [00:00<00:00, 24265.75it/s]


### COMPUTE THE MIN AND MAXES FOR THE SELECTED ENSMEMLES

Some important points:

1. For now we are working in pixel absolute values for each cell.  

2. A better option might be to find the maximum peak of a cell during a window and then save that value and normalize all future events by that value. (note: any online BMI filtering/chagnig of data will need to account for this).


In [6]:
# ######################################################################
# ########### RECOMPUTE TRACES WITH SINGLE FRAME PRECISION #############
# ######################################################################
# # NOT SURE THIS IS NECESSARY... TO MODIFY so we can work withs subsampled series al
gr.trace_subsample = 1             # Subsample the time series to go faster;

# visualize traces
gr.compute_traces2()


memmap :  (20000, 512, 512)


100%|████████████████████████████████████████████████████████████████████████████| 20000/20000 [00:34<00:00, 575.84it/s]


In [12]:
#############################################
########### RUN THRSHOLD SETTER #############
#############################################

# 
gr.sample_rate = 30
gr.post_reward_lockout = 10   # reward lockout in seconds
gr.rois_smooth_window = 10     # of frames to use to smooth the realtime signal
gr.smooth_diff_function_flag = True    # use a kernel window to smooth current value

#
gr.balance_ensemble_rewards_flag = True   #this makes sure that both ensembles elicit a similar number of random rewards


# find 30% reward threshold
gr.find_reward_thresholds_low_and_high()

#gr.find_reward_thresholds_high()  # this only rewards when sound passes specific level
print ("thresholds: ", gr.high)

########################################
gr.plot_rewarded_ensembles()



100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████| 9990/9990 [00:00<00:00, 49822.06it/s]


low, high:  -2060.456470588235 1863.7431724653948
nsec recording:  333 max # of random rewards (i.e. every 30sec)  11
updated rwards #:  11 -701.7124065193985 703.2902003303992
thresholds:  703.2902003303992


In [13]:
#############################################
########### RUN THRSHOLD SETTER #############
#############################################

# save all data to disk
# also add the tone values here as well that will be used for the experiment
gr.low_freq = 2000
gr.high_freq = 18000

# save cell pixel locations as 2 column array inside list
cells = []
for k in range(4):
    temp = gr.indexes[both[k]]
    temp1 = temp[0]
    temp2 = temp[1]
    temp = np.vstack((temp1,temp2))

    cells.append(temp)

# save individual pixels of each cell - currently implemented in BMI
np.savez(os.path.join(os.path.split(fname)[0],
                        'rois_pixels_and_thresholds.npz'),
            cell0 = cells[0],
            cell1 = cells[1],
            cell2 = cells[2],
            cell3 = cells[3],
            cell_f0s = gr.roi_f0s,
            cell_centres = np.int32(gr.rois)[both],
            cell_ids = both,
            all_rois = np.int32(gr.rois),
            low_threshold = gr.low,
            high_threshold = gr.high,
            low_freq = gr.low_freq,
            high_freq = gr.high_freq,
            cell_traces = gr.roi_traces,
            sample_rate = gr.sample_rate,
            post_reward_lockout = gr.post_reward_lockout,
            balance_ensemble_rewards_flag = gr.balance_ensemble_rewards_flag,
            rois_smooth_window = gr.rois_smooth_window,
            smooth_diff_function_flag = gr.smooth_diff_function_flag
         
        )



In [9]:
print (os.path.join(os.path.split(fname)[0],
                        'rois_pixels_and_thresholds.npz'))

D:\bmi\mouse1\rois_pixels_and_thresholds.npz


In [ ]:
#######################################################
######### MANUAL ROI SELECTOR - DO NOT DELETE #########
#######################################################

# # importing the module
# import cv2

# # function to display the coordinates of
# # of the points clicked on the image
# def click_event(event, x, y, flags, params):

#     # checking for left mouse clicks
#     if event == cv2.EVENT_LBUTTONDOWN:

#         # displaying the coordinates
#         # on the Shell
#         print(x, ' ', y)
#         rois_manual.append([x,y])

#         # displaying the coordinates
#         # on the image window
#         #font = cv2.FONT_HERSHEY_SIMPLEX
#         img[y-2:y+3, x-2:x+3] = 0
   
#         #cv2.putText(img, str(x) + ',' +
#         #            str(y), (x,y), font,
#         #            .3, (255, 0, 0), 2)
#         cv2.imshow('image', img)

#     # checking for right mouse clicks	
#     #if event==cv2.EVENT_RBUTTONDOWN:
#     #    
#     #    np.save()

# # driver function
# if __name__=="__main__":
    
#     global rois_manual
    
#     rois_manual = []
    
#     # reading the image
#     #img = cv2.imread('lena.jpg', 1)
#     img = std_map.copy()
    
#     img = cv2.resize(img, (int(img.shape[0]*1.5),
#                            int(img.shape[1]*1.5))) 

#     # displaying the image
#     cv2.imshow('image', img)

#     # setting mouse handler for the image
#     # and calling the click_event() function
#     cv2.setMouseCallback('image', click_event)

#     # wait for a key to be pressed to exit
#     cv2.waitKey(0)

#     # close the window
#     cv2.destroyAllWindows()

# print (" DONE LABELING: ")
# print ("ROIS: ", rois_manual)



In [2]:
data = np.load('/media/cat/4TB/donato/BSCOPE_tests/rois_pixels_and_thresholds.npz')

low_thresh = data['low_threshold']
high_thresh = data['high_threshold']

print (low_thresh, high_thresh)



-768.7697339352011 546.5230278373468


In [10]:

octave_size = 0.25

# 
def get_octave_frequencies(low_freq,
                  high_freq,
                  octave_size):
    
    #
    octaves = []
    
    #
    octaves.append(low_freq)
    temp = low_freq
    while True:
        temp = temp * (1+octave_size)
        if temp>high_freq:
            break
        octaves.append(temp)

    return octaves
#
octaves = get_octave_frequencies(2000,
              18000,
              octave_size)
print (len(octaves),octaves)
      
    


10 [2000, 2500.0, 3125.0, 3906.25, 4882.8125, 6103.515625, 7629.39453125, 9536.7431640625, 11920.928955078125, 14901.161193847656]


In [3]:
#################################################################
#################################################################
#################################################################
import tkinter as tk
from tkinter.filedialog import askopenfilename

# bad practice!!! do not use global varaiables!!!
global filename
# 
root= tk.Tk()
root.geometry('1250x200+1250+200')

#
def get_calibration_raw_data():
    global filename
    #
    filename = askopenfilename()
    print('Selected 3:', filename)
    root.destroy()
    
    
# 
button1 = tk.Button(text='Load calibration .raw', 
                    command=get_calibration_raw_data, 
                    bg='brown', 
                    fg='white')
button1.pack(padx=2, pady=5)

# 
root.mainloop()
print (filename)


Selected 3: /home/cat/code/bmi/bmi_command_line.py
/home/cat/code/bmi/bmi_command_line.py
